# Phase 6 — Extra experiments: RQ4, capacity control, RQ3 and the boundary analysis
Phase 5 told us *how much* noise hurts each model and *what kind* of damage it does. This phase asks three follow-up questions that make the paper's conclusions trustworthy and useful:

| Part | Question | Why it matters |
|---|---|---|
| **A. RQ4: regularisation** | Do standard fixes (slower learning, early stopping, smaller bootstrap samples) repair the damage, and does that depend on the noise type? | Turns findings into **practical advice** |
| **B. Capacity control** | Phase 5 found boosting *more* robust than bagging. Is that because of the ensemble method, or just because boosting used **smaller trees**? | Prevents a **wrong conclusion** a reviewer would catch |
| **C. RQ3: sources of variance** | When noise makes a model unstable, is it because of *which* labels were flipped or because of *which* training rows it saw? | Explains the **mechanism** |
| **D. Boundary analysis** | Is the damage concentrated on test points near the decision boundary? | Connects the noise types to *where* errors appear |

All runs use the same 15 datasets, seeds and R = 51 as Phase 5. Thanks to common random numbers, every new run is **paired** with the matching Phase 5 run: it saw exactly the same training samples and flipped labels.

Kernel: **Python (labelnoise)**.

## Step 0 — Run (or resume) the experiments
The work is done by **`src/run_extra.py`**, resumable just like Phase 5. It was started from a terminal:
```
conda run -n labelnoise python src/run_extra.py
```
Calling it here finishes instantly if everything is already saved.

| Experiment | Models | Conditions | Rows |
|---|---|---|---|
| rq4 | GB_reg, XGB_reg, AdaBoost_reg, Bagging_small | clean + 3 types × {10, 20, 30 %} | 600 |
| capacity | Bagging_d3, RF_d3, AdaBoost_d3, XGB_d3, GB_d8, XGB_d8 | same | 900 |
| rq3 | the 6 original models × 3 designs | 3 types at 20 % | 810 |

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

NB_DIR = Path.cwd()
BASE = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
sys.path.insert(0, str(BASE / "src"))
from core import (MODEL_NAMES, MODEL_FAMILY, NOISE_TYPES, REG_MODELS, CAPACITY_MODELS, TREE_DEPTH,
                  load_dataset, decompose)
import run_extra
for exp in ["rq4", "capacity", "rq3"]:
    run_extra.EXPERIMENTS[exp]()

RAW, TABLES = BASE / "results" / "raw", BASE / "results" / "tables"
TYPES = ["symmetric", "one_sided", "boundary"]
def load(folder):
    return pd.concat([pd.read_csv(f) for f in sorted((RAW / folder).glob("*.csv"))], ignore_index=True)

main = pd.read_csv(BASE / "results" / "main_results.csv")
rq4, cap, rq3 = load("rq4"), load("capacity"), load("rq3")
for name, df, n in [("rq4", rq4, 600), ("capacity", cap, 900), ("rq3", rq3, 810)]:
    ok = len(df) == n and df.identity_gap.abs().max() < 1e-12 and not df.isna().any().any()
    print(f"{'OK  ' if ok else 'FAIL'}  {name}: {len(df)} rows (expected {n}), identity and completeness checked")

11:15:34  START rq4: 4 models x 10 conditions, R = 51


11:15:35  DONE rq4


11:15:35  START capacity: 6 models x 10 conditions, R = 51


11:15:36  DONE capacity


11:15:36  START rq3: 3 designs x 6 models x 3 noise types, rho = 0.2, R = 51


11:15:37  DONE rq3


OK    rq4: 600 rows (expected 600), identity and completeness checked
OK    capacity: 900 rows (expected 900), identity and completeness checked
OK    rq3: 810 rows (expected 810), identity and completeness checked


## Helper: damage relative to clean labels
As in Phase 5, **damage** = value under noise − value on clean labels, for the same dataset and model. `deltas()` adds Δ error, Δ bias and Δ net var to any results table.

In [2]:
def deltas(df):
    clean = df[df.noise_type == "none"].set_index(["dataset", "model"])
    noisy = df[df.noise_type != "none"].copy()
    key = list(zip(noisy.dataset, noisy.model))
    for col in ["error", "bias", "net_var"]:
        noisy["d_" + col] = noisy[col].to_numpy() - clean.loc[key, col].to_numpy()
    noisy["clean_error"] = clean.loc[key, "error"].to_numpy()
    return noisy

d_main = deltas(main)

## Part A — RQ4: do common fixes repair the damage?
Each regularised model is compared with its base model **on the same dataset, noise type and rate**:

| Regularised | Base | What the fix does |
|---|---|---|
| GB_reg | GB | Smaller learning rate (0.05), uses 80 % of rows per tree, stops early when a held-out 10 % stops improving |
| XGB_reg | XGB | Same idea: learning rate 0.05, 80 % row sampling, early stopping on 10 % of the (noisy) training data |
| AdaBoost_reg | AdaBoost | Learning rate 0.5: each round's correction is halved |
| Bagging_small | Bagging | Each tree sees only 20 % of the rows (Sabzevari et al., 2015) |

**What we report** (averages over 15 datasets):
- **noisy error, base vs fixed**: what a practitioner actually gets
- **damage recovered** = 1 − Δerror(fixed) / Δerror(base): the share of the *noise damage* the fix removes
- **clean cost** = clean error(fixed) − clean error(base): what the fix costs when labels are clean
- **p-value**: Wilcoxon signed-rank test over the 15 datasets (paired), on the noisy error. Below 0.05 means the difference is unlikely to be chance.

In [3]:
d_rq4 = deltas(rq4)
rows = []
for fixed, base in REG_MODELS.items():
    for nt in TYPES:
        for rho in [0.1, 0.2, 0.3]:
            b = d_main[(d_main.model == base) & (d_main.noise_type == nt) & np.isclose(d_main.rho, rho)].set_index("dataset")
            f = d_rq4[(d_rq4.model == fixed) & (d_rq4.noise_type == nt) & np.isclose(d_rq4.rho, rho)].set_index("dataset")
            f = f.loc[b.index]
            diff = f.error - b.error
            p = wilcoxon(f.error, b.error).pvalue if (diff != 0).any() else 1.0
            rows.append({"fix": fixed, "noise_type": nt, "rho": rho,
                         "noisy error base %": 100 * b.error.mean(), "noisy error fixed %": 100 * f.error.mean(),
                         "damage base (pp)": 100 * b.d_error.mean(), "damage fixed (pp)": 100 * f.d_error.mean(),
                         "damage recovered %": 100 * (1 - f.d_error.mean() / b.d_error.mean()),
                         "clean cost (pp)": 100 * (f.clean_error - b.clean_error).mean(),
                         "fixed better on # datasets": int((diff < 0).sum()), "p (Wilcoxon)": p})
rq4_tab = pd.DataFrame(rows)
rq4_tab.to_csv(TABLES / "phase6_rq4_regularisation.csv", index=False)
rq4_tab[rq4_tab.rho == 0.2].round(3).set_index(["fix", "noise_type"])

rho  noisy error base %  noisy error fixed %  \
fix           noise_type                                                 
GB_reg        symmetric   0.2              17.578               16.653   
              one_sided   0.2              16.114               16.083   
              boundary    0.2              18.325               17.778   
XGB_reg       symmetric   0.2              17.303               16.185   
              one_sided   0.2              15.468               15.289   
              boundary    0.2              18.451               17.664   
AdaBoost_reg  symmetric   0.2              17.656               17.817   
              one_sided   0.2              18.233               18.827   
              boundary    0.2              17.864               18.140   
Bagging_small symmetric   0.2              18.774               15.953   
              one_sided   0.2              15.464               15.431   
              boundary    0.2              20.079               17.995   

                          damage base (pp)  damage fixed (pp)  \
fix           noise_type                                        
GB_reg        symmetric              3.567              2.895   
              one_sided              2.103              2.325   
              boundary               4.314              4.020   
XGB_reg       symmetric              3.775              3.043   
              one_sided              1.939              2.148   
              boundary               4.923              4.522   
AdaBoost_reg  symmetric              2.284              1.462   
              one_sided              2.861              2.471   
              boundary               2.492              1.784   
Bagging_small symmetric              5.160              2.267   
              one_sided              1.850              1.745   
              boundary               6.464              4.309   

                          damage recovered %  clean cost (pp)  \
fix           noise_type                                        
GB_reg        symmetric               18.840           -0.253   
              one_sided              -10.565           -0.253   
              boundary                 6.809           -0.253   
XGB_reg       symmetric               19.372           -0.387   
              one_sided              -10.749           -0.387   
              boundary                 8.145           -0.387   
AdaBoost_reg  symmetric               36.008            0.984   
              one_sided               13.624            0.984   
              boundary                28.397            0.984   
Bagging_small symmetric               56.063            0.072   
              one_sided                5.666            0.072   
              boundary                33.338            0.072   

                          fixed better on # datasets  p (Wilcoxon)  
fix           noise_type                                            
GB_reg        symmetric                           12         0.041  
              one_sided                            9         0.489  
              boundary                            12         0.041  
XGB_reg       symmetric                           13         0.015  
              one_sided                            9         0.151  
              boundary                            11         0.030  
AdaBoost_reg  symmetric                            8         0.421  
              one_sided                            5         0.330  
              boundary                             8         0.847  
Bagging_small symmetric                           14         0.003  
              one_sided                            8         0.359  
              boundary                            14         0.004

**How to read it:**
- **Damage recovered** near 100 % means the fix removes almost all the harm noise caused; near 0 % means it does nothing; negative means it makes the damage worse.
- Compare **the same fix across noise types**. Our hypothesis H5 (from Long & Servedio, 2010) predicts that fixes help less under *one-sided* noise, where the damage is mostly bias, than under symmetric noise, where it is mostly variance. Variance-reducing fixes should not cure a wrong rule.
- **Clean cost** tells you whether the fix is "free": a fix that loses accuracy on clean data is only worth it when you expect noise.

Damage recovered at every rate, to check whether the pattern holds as noise grows:

In [4]:
rq4_tab.pivot_table(index=["fix", "noise_type"], columns="rho", values="damage recovered %").round(0) \
       .reindex(pd.MultiIndex.from_product([list(REG_MODELS), TYPES]))

rho                       0.1   0.2   0.3
GB_reg        symmetric   7.0  19.0  32.0
              one_sided  -1.0 -11.0 -12.0
              boundary   -2.0   7.0  12.0
XGB_reg       symmetric   9.0  19.0  30.0
              one_sided -22.0 -11.0 -12.0
              boundary    6.0   8.0  10.0
AdaBoost_reg  symmetric  37.0  36.0  35.0
              one_sided  19.0  14.0  -4.0
              boundary   18.0  28.0  29.0
Bagging_small symmetric  64.0  56.0  43.0
              one_sided   7.0   6.0   1.0
              boundary   39.0  33.0  24.0

## Part B — Capacity control: the ensemble method, or the tree size?
In Phase 5, Bagging and RF used **full-depth** trees while AdaBoost used **stumps** (depth 1) and GB/XGB depth 3–4. Deep trees can memorise individual wrong labels; shallow trees cannot. So the Phase 5 comparison mixed two things together.

Here every family is also run at **depth 3** (a fair comparison), and the boosting models also at **depth 8** (deep boosting, like Dietterich, 2000). The table shows, for each model: tree depth, clean error, damage at 20 % noise, and the variance share of that damage.

In [5]:
d_cap = deltas(cap)
both = pd.concat([d_main, d_cap], ignore_index=True)
order = ["Tree", "Bagging", "RF", "Bagging_d3", "RF_d3",
         "AdaBoost", "AdaBoost_d3", "GB", "XGB_d3", "XGB", "GB_d8", "XGB_d8"]
s = both[np.isclose(both.rho, 0.2)].groupby(["model", "noise_type"])[["clean_error", "d_error", "d_net_var"]].mean()
cap_tab = pd.DataFrame({
    "family": [MODEL_FAMILY[m] for m in order],
    "depth": [TREE_DEPTH[m] if TREE_DEPTH[m] else "full" for m in order],
    "clean error %": [100 * s.loc[(m, "symmetric"), "clean_error"] for m in order],
}, index=order)
for nt in TYPES:
    cap_tab[f"damage {nt} (pp)"] = [100 * s.loc[(m, nt), "d_error"] for m in order]
for nt in TYPES:
    cap_tab[f"var share {nt} %"] = [100 * s.loc[(m, nt), "d_net_var"] / s.loc[(m, nt), "d_error"] for m in order]
cap_tab.to_csv(TABLES / "phase6_capacity.csv")
cap_tab.round(1)

,family,depth,clean error %,damage symmetric (pp),damage one_sided (pp),damage boundary (pp),var share symmetric %,var share one_sided %,var share boundary %
Tree,single,full,17.0,10.9,4.2,8.6,100.8,94.2,54.4
Bagging,parallel,full,13.6,5.2,1.8,6.5,105.4,88.5,49.0
RF,parallel,full,13.3,4.7,2.4,6.1,98.6,36.2,44.1
Bagging_d3,parallel,3,18.0,0.2,0.7,1.3,75.8,-61.1,41.5
RF_d3,parallel,3,17.7,0.4,3.2,1.1,94.9,3.5,63.3
AdaBoost,sequential,1,15.4,2.3,2.9,2.5,71.3,7.4,36.7
AdaBoost_d3,sequential,3,14.1,5.2,2.6,5.4,102.3,43.2,47.5
GB,sequential,3,14.0,3.6,2.1,4.3,97.6,16.5,39.9
XGB_d3,sequential,3,14.0,2.9,1.8,4.0,102.6,14.9,30.2
XGB,sequential,4,13.5,3.8,1.9,4.9,95.6,39.9,38.7


**The key comparison: all families at the same depth (3).** If parallel and sequential ensembles are equally damaged at equal depth, the Phase 5 result was caused by **tree size**, not by the ensemble method. Wilcoxon tests compare the damage of each depth-3 parallel model against each depth-3 sequential model over the 15 datasets (20 % noise):

In [6]:
def paired(m1, m2, nt, rho=0.2, col="d_error"):
    a = both[(both.model == m1) & (both.noise_type == nt) & np.isclose(both.rho, rho)].set_index("dataset")[col]
    b = both[(both.model == m2) & (both.noise_type == nt) & np.isclose(both.rho, rho)].set_index("dataset")[col].loc[a.index]
    return a.mean() - b.mean(), wilcoxon(a, b).pvalue if (a != b).any() else 1.0

rows = []
for par in ["Bagging_d3", "RF_d3"]:
    for seq in ["AdaBoost_d3", "GB", "XGB_d3"]:
        for nt in TYPES:
            diff, p = paired(par, seq, nt)
            rows.append({"parallel (d3)": par, "sequential (d3)": seq, "noise_type": nt,
                         "damage difference (pp), parallel − sequential": 100 * diff, "p (Wilcoxon)": p})
eq = pd.DataFrame(rows)
eq.to_csv(TABLES / "phase6_capacity_equal_depth_tests.csv", index=False)
eq.round(3)

,parallel (d3),sequential (d3),noise_type,"damage difference (pp), parallel − sequential",p (Wilcoxon)
0,Bagging_d3,AdaBoost_d3,symmetric,-5.024,0.000
1,Bagging_d3,AdaBoost_d3,one_sided,-1.991,0.030
2,Bagging_d3,AdaBoost_d3,boundary,-4.182,0.000
3,Bagging_d3,GB,symmetric,-3.378,0.000
4,Bagging_d3,GB,one_sided,-1.448,0.008
5,Bagging_d3,GB,boundary,-3.063,0.000
6,Bagging_d3,XGB_d3,symmetric,-2.690,0.000
7,Bagging_d3,XGB_d3,one_sided,-1.172,0.035
8,Bagging_d3,XGB_d3,boundary,-2.791,0.000
9,RF_d3,AdaBoost_d3,symmetric,-4.834,0.000


**Depth effect within each family:** how damage changes when only the tree size changes (20 % noise, averaged over noise types):

In [7]:
pairs = [("Bagging", "Bagging_d3"), ("RF", "RF_d3"), ("AdaBoost", "AdaBoost_d3"),
         ("GB", "GB_d8"), ("XGB_d3", "XGB_d8")]
rows = []
for a, b in pairs:
    for nt in TYPES:
        diff, p = paired(b, a, nt)
        rows.append({"from": f"{a} (depth {TREE_DEPTH[a] or 'full'})", "to": f"{b} (depth {TREE_DEPTH[b] or 'full'})",
                     "noise_type": nt, "change in damage (pp)": 100 * diff, "p (Wilcoxon)": p})
depth_tab = pd.DataFrame(rows)
depth_tab.to_csv(TABLES / "phase6_capacity_depth_effect.csv", index=False)
depth_tab.round(3)

,from,to,noise_type,change in damage (pp),p (Wilcoxon)
0,Bagging (depth full),Bagging_d3 (depth 3),symmetric,-4.971,0.000
1,Bagging (depth full),Bagging_d3 (depth 3),one_sided,-1.194,0.041
2,Bagging (depth full),Bagging_d3 (depth 3),boundary,-5.213,0.000
3,RF (depth full),RF_d3 (depth 3),symmetric,-4.294,0.000
4,RF (depth full),RF_d3 (depth 3),one_sided,0.811,0.524
5,RF (depth full),RF_d3 (depth 3),boundary,-5.026,0.000
6,AdaBoost (depth 1),AdaBoost_d3 (depth 3),symmetric,2.929,0.000
7,AdaBoost (depth 1),AdaBoost_d3 (depth 3),one_sided,-0.215,0.359
8,AdaBoost (depth 1),AdaBoost_d3 (depth 3),boundary,2.941,0.000
9,GB (depth 3),GB_d8 (depth 8),symmetric,2.088,0.026


**How to read Part B:**
- The **equal-depth tests** answer the reviewer's question directly: is boosting really more robust, or did it just have smaller trees?
- The **depth-effect table** shows how much tree size alone changes the damage. If deep boosting (GB_d8, XGB_d8) is hurt much more than shallow boosting, that reconciles our results with Dietterich's.
- Always look at **clean error** too: a very shallow model may be hurt less by noise simply because it was already worse on clean data (it underfits). The fair question is which model has the lowest error *under noise*.

## Part C — RQ3: where does the noise-induced variance come from?
At 20 % noise, each model was trained 51 times under three designs, each re-drawing only some sources of randomness:

| Design | Re-drawn each round | Held fixed | Variance it measures |
|---|---|---|---|
| **M**: model only | the model's own random seed | training rows, flipped labels | internal randomness of the algorithm |
| **A**: noise only | which labels are flipped (+ seed) | training rows | **noise realisation** |
| **B**: sampling only | which training rows (+ seed) | flipped labels | **sampling** |
| **C**: both (Phase 5) | everything | nothing | total |

**Variance** here is the total (unbiased + biased): how much the 51 models disagree. Then:
- **A − M** = variance added by *which labels happen to be wrong*
- **B − M** = variance added by *which rows happen to be in the training set*

In [8]:
rq3 = rq3.assign(var=rq3.var_unbiased + rq3.var_biased)
c = main[np.isclose(main.rho, 0.2) & main.noise_type.isin(TYPES)].assign(
        var=lambda d: d.var_unbiased + d.var_biased, design="both (Phase 5)")
allv = pd.concat([rq3, c], ignore_index=True)
v = allv.pivot_table(index=["noise_type", "model"], columns="design", values="var", aggfunc="mean")
v = v[["model_only", "noise_only", "sampling_only", "both (Phase 5)"]] * 100
v["noise adds (A−M)"] = v.noise_only - v.model_only
v["sampling adds (B−M)"] = v.sampling_only - v.model_only
v = v.reindex(pd.MultiIndex.from_product([TYPES, MODEL_NAMES]))
v.to_csv(TABLES / "phase6_rq3_variance_sources.csv")
print("Average variance (%) by design, 20 % noise")
v.round(2)

Average variance (%) by design, 20 % noise


design              model_only  noise_only  sampling_only  both (Phase 5)  \
symmetric Tree            5.19       25.62          24.57           24.00   
          Bagging         3.95       12.98          13.00           13.08   
          RF              3.87       11.65          11.14           11.66   
          AdaBoost        0.00        8.26           9.14            8.97   
          GB              0.25        9.73          10.66           10.49   
          XGB             0.00       10.21          10.98           10.76   
one_sided Tree            2.90       14.11          14.38           15.74   
          Bagging         1.87        6.02           7.03            7.41   
          RF              2.13        5.37           5.81            6.31   
          AdaBoost        0.02        4.34           5.32            5.63   
          GB              0.27        4.83           5.92            6.20   
          XGB             0.00        4.98           6.06            6.34   
boundary  Tree            4.01       21.09          19.24           20.48   
          Bagging         3.13       12.75          10.63           13.19   
          RF              3.40       11.54           9.22           11.77   
          AdaBoost        0.00        7.83           7.92            8.38   
          GB              0.43        9.69           8.99           10.30   
          XGB             0.00       10.24           9.38           10.87   

design              noise adds (A−M)  sampling adds (B−M)  
symmetric Tree                 20.43                19.38  
          Bagging               9.03                 9.05  
          RF                    7.78                 7.26  
          AdaBoost              8.26                 9.14  
          GB                    9.48                10.41  
          XGB                  10.21                10.98  
one_sided Tree                 11.21                11.48  
          Bagging               4.15                 5.16  
          RF                    3.24                 3.68  
          AdaBoost              4.33                 5.30  
          GB                    4.56                 5.66  
          XGB                   4.98                 6.06  
boundary  Tree                 17.07                15.23  
          Bagging               9.61                 7.50  
          RF                    8.14                 5.83  
          AdaBoost              7.83                 7.92  
          GB                    9.26                 8.56  
          XGB                  10.24                 9.38

**How to read Part C:**
- If **noise adds** is larger than **sampling adds**, a model's instability under noise comes mainly from *which labels are wrong*. Then even collecting more data from the same noisy process wouldn't make it stable; only fewer wrong labels would.
- If **sampling adds** dominates, the instability is ordinary sampling variance that noise amplifies. That is exactly what averaging (bagging) is designed to remove.
- Compare **parallel vs sequential** models: bagging averages over bootstrap samples, so it should remove sampling variance well, but it cannot average away a noise mask that every tree shares.

## Part D — Is the damage concentrated near the decision boundary?
We reuse the **saved Phase 5 predictions** (no new training). For every dataset, a logistic regression trained on the clean training labels scores each **test** point; its **margin** |P(class 1) − 0.5| says how close the point is to the boundary. We split the test points into the **nearer half** and the **farther half** and decompose the error separately in each half.

If boundary noise mostly damages near-boundary test points, while symmetric noise damages both halves, the noise types differ not only in *how much* and *what kind* of damage but also in **where** it lands.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
SEED = json.loads((BASE / "config.json").read_text())["SEED"]
PRED = RAW / "main_preds"

rows = []
for ds in sorted(main.dataset.unique()):
    X_tr, y_tr, X_te, y_te = load_dataset(ds, BASE / "data")
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=SEED)).fit(X_tr, y_tr)
    margin = np.abs(lr.predict_proba(X_te)[:, 1] - 0.5)
    near = margin <= np.median(margin)
    for model in MODEL_NAMES:
        for nt, rho in [("none", 0.0)] + [(t, 0.2) for t in TYPES]:
            P = np.load(PRED / ds / f"{model}__{nt}__{rho:.2f}.npz")["P"]
            for half, mask in [("near", near), ("far", ~near)]:
                d = decompose(P[:, mask], y_te[mask])
                rows.append({"dataset": ds, "model": model, "noise_type": nt, "half": half,
                             "error": d["error"], "bias": d["bias"], "net_var": d["net_var"]})
mh = pd.DataFrame(rows)
clean_h = mh[mh.noise_type == "none"].set_index(["dataset", "model", "half"])
noisy_h = mh[mh.noise_type != "none"].copy()
k = list(zip(noisy_h.dataset, noisy_h.model, noisy_h.half))
for col in ["error", "bias", "net_var"]:
    noisy_h["d_" + col] = noisy_h[col].to_numpy() - clean_h.loc[k, col].to_numpy()
noisy_h.to_csv(TABLES / "phase6_margin_halves_raw.csv", index=False)

mt = (noisy_h.groupby(["noise_type", "model", "half"])["d_error"].mean().unstack("half") * 100)
mt["near / far ratio"] = mt["near"] / mt["far"]
mt = mt.reindex(pd.MultiIndex.from_product([TYPES, MODEL_NAMES]))
mt.to_csv(TABLES / "phase6_margin_halves.csv")
print("Damage (pp) at 20 % noise in the test half nearer to / farther from the boundary")
mt.round(2)

Damage (pp) at 20 % noise in the test half nearer to / farther from the boundary


half                  far   near  near / far ratio
symmetric Tree      12.56   9.21              0.73
          Bagging    5.12   5.20              1.02
          RF         4.32   5.03              1.17
          AdaBoost   1.81   2.76              1.52
          GB         3.15   3.98              1.26
          XGB        3.01   4.54              1.51
one_sided Tree       3.98   4.38              1.10
          Bagging    1.34   2.36              1.76
          RF         1.58   3.14              1.99
          AdaBoost   1.19   4.53              3.82
          GB         0.91   3.30              3.63
          XGB        0.82   3.05              3.72
boundary  Tree       3.65  13.54              3.71
          Bagging    1.15  11.77             10.25
          RF         1.02  11.21             11.03
          AdaBoost   0.26   4.71             17.94
          GB         0.78   7.83              9.99
          XGB        0.84   8.99             10.70

**How to read Part D:** a **near/far ratio** well above 1 means the damage is concentrated on hard, borderline test points; a ratio near 1 means it is spread evenly. Comparing the three noise types shows *where* each kind of label error hurts.

## Phase 6 checklist
- [ ] All three experiments complete (Step 0 prints OK three times)
- [ ] Part A: RQ4 table read, H5 judged (do fixes help less under one-sided noise?)
- [ ] Part B: capacity question answered (ensemble method vs tree size)
- [ ] Part C: RQ3 answered (noise realisation vs sampling)
- [ ] Part D: boundary concentration checked
- [ ] Your own notes on each part for the Results section

**Next: Phase 6B** (optional multiclass check) **or Phase 7** (statistics and figures for the paper).